# Interest-rate implied volatility surfaces

This notebook plots **Black–76**, **SABR**, **LMM** (piecewise caplet structure), and **Gaussian HJM / Hull–White** implied-volatility style surfaces.

Install from the repo root: `pip install -e .` then `pip install jupyter` (or `pip install -e ".[notebook]"` if you use the optional extra).

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

from rates_models.black76 import black76_price, black76_implied_vol
from rates_models.sabr import sabr_implied_vol_lognormal
from rates_models.lmm import LMMConfig
from rates_models.hjm import caplet_black_vol_from_gaussian_hjm

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")
%matplotlib inline

## 1. Black–76 (flat lognormal vol)

Black–76 does not **generate** a smile by itself: quoted **Black vol** $\sigma_B$ is the input. Here we show a **flat** surface $\sigma_B(K, T) = \text{const}$ (and a second panel: implied vol recovered from synthetic prices, which is numerically flat).

In [ ]:
forward = 0.03
sigma_flat = 0.25
# Strikes not too far from forward so implied-vol inversion stays well bracketed
strikes = np.linspace(0.018, 0.048, 45)
expiries = np.linspace(0.25, 5.0, 45)
K, T = np.meshgrid(strikes, expiries)
Z_black = np.full_like(K, sigma_flat, dtype=float)

fig = plt.figure(figsize=(6.5, 5))
ax = fig.add_subplot(1, 1, 1, projection="3d")
surf = ax.plot_surface(K, T, Z_black, cmap=cm.viridis, linewidth=0, antialiased=True, alpha=0.95)
ax.set_xlabel("Strike (rate)")
ax.set_ylabel("Expiry (years)")
ax.set_zlabel("Black vol")
ax.set_title("Black–76: prescribed flat σ(K, T)")
fig.colorbar(surf, ax=ax, shrink=0.55, label="σ")
plt.tight_layout()
plt.show()

vprice = np.vectorize(black76_price, otypes=[float])
vimplied = np.vectorize(black76_implied_vol, otypes=[float])
px = vprice(forward, K, T, sigma_flat)
iv_grid = vimplied(forward, K, T, px)

fig, ax = plt.subplots(figsize=(7, 5))
c = ax.contourf(K, T, iv_grid, levels=30, cmap="viridis")
ax.set_xlabel("Strike (rate)")
ax.set_ylabel("Expiry (years)")
ax.set_title("Black–76: implied vol inverted from prices (flat σ in → flat σ out)")
plt.colorbar(c, ax=ax, label="Implied Black vol")
plt.tight_layout()
plt.show()

## 2. SABR smile across strike and expiry

Hagan et al. **lognormal** implied vol $\sigma_B(K, T)$ for fixed forward. Parameters can vary with $T$; here $\alpha(T)$ scales mildly with horizon for a more realistic term structure.

In [ ]:
f0 = 0.03
beta, rho, nu = 0.4, -0.25, 0.35

def alpha_of_t(t: float) -> float:
    return 0.015 + 0.006 * np.sqrt(np.maximum(t, 0.05))

strikes_s = np.linspace(0.01, 0.055, 50)
expiries_s = np.linspace(0.25, 5.0, 50)
Ks, Ts = np.meshgrid(strikes_s, expiries_s)
Z_sabr = np.vectorize(lambda k, t: sabr_implied_vol_lognormal(f0, k, t, alpha_of_t(t), beta, rho, nu))(
    Ks, Ts
)

fig = plt.figure(figsize=(11, 5))
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
surf = ax1.plot_surface(Ks, Ts, Z_sabr, cmap=cm.plasma, linewidth=0, antialiased=True)
ax1.set_xlabel("Strike (rate)")
ax1.set_ylabel("Expiry (years)")
ax1.set_zlabel("Black vol")
ax1.set_title("SABR: Black implied vol surface")
fig.colorbar(surf, ax=ax1, shrink=0.55, label="σ")

ax2 = fig.add_subplot(1, 2, 2)
cf = ax2.contourf(Ks, Ts, Z_sabr, levels=28, cmap="plasma")
ax2.set_xlabel("Strike (rate)")
ax2.set_ylabel("Expiry (years)")
ax2.set_title("Contour view")
fig.colorbar(cf, ax=ax2, label="σ")
plt.tight_layout()
plt.show()

## 3. LMM (piecewise caplet vols → surface)

The discrete **LIBOR market model** in the repo uses a **constant** $\sigma_i$ per forward. That implies an **ATM caplet** Black vol curve vs fixing time, but **no strike smile** unless you extend the model. Here we plot $\sigma(T_\text{fix}, K)$ as **constant in $K$** within each tenor bucket (a **step** in fixing time), which matches a standard lognormal LMM without stochastic vol.

In [ ]:
n = 12
tau = np.full(n, 0.25)
t_dates = np.concatenate([[0.0], np.cumsum(tau)])  # T_0..T_n
fixing_times = t_dates[:-1]  # T_k for LIBOR k
vols = np.clip(0.18 + 0.012 * np.arange(n) + 0.03 * np.sin(np.linspace(0, 3, n)), 0.08, None)

strikes_l = np.linspace(0.01, 0.055, 60)
t_axis = np.linspace(0.05, t_dates[-1] - 1e-6, 80)
Klm, Tlm = np.meshgrid(strikes_l, t_axis)

def vol_lmm_piecewise(t: float) -> float:
    idx = min(max(np.searchsorted(t_dates, t, side="right") - 1, 0), n - 1)
    return float(vols[idx])

Z_lmm = np.vectorize(lambda k, t: vol_lmm_piecewise(t))(Klm, Tlm)

fig = plt.figure(figsize=(11, 5))
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
surf = ax1.plot_surface(Klm, Tlm, Z_lmm, cmap=cm.cividis, linewidth=0, antialiased=True)
ax1.set_xlabel("Strike (rate)")
ax1.set_ylabel("Caplet fixing time (years)")
ax1.set_zlabel("ATM Black vol (bucket)")
ax1.set_title("LMM-style piecewise caplet vol (flat in strike)")
fig.colorbar(surf, ax=ax1, shrink=0.55)

ax2 = fig.add_subplot(1, 2, 2)
ax2.step(t_dates[:-1], vols, where="post", label="σ per forward")
ax2.set_xlabel("Tenor start T_k")
ax2.set_ylabel("σ")
ax2.set_title("Piecewise vols (LMMConfig.vols)")
ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.show()

_ = LMMConfig(np.full(n, 0.03), tau, vols, np.eye(n))  # same numbers as config

## 4. Gaussian HJM / HW caplet Black vol

`caplet_black_vol_from_gaussian_hjm(T, S, σ₀, κ)` gives the **Black** vol for a caplet resetting at $T$ on an accrual ending at $S$, under one-factor Gaussian HJM with $\sigma(t,T)=\sigma_0 e^{-\kappa(T-t)}$. Axes: **reset time** $T$ and **period length** $\tau=S-T$.

In [ ]:
sigma0, kappa = 0.012, 0.04
t_reset = np.linspace(0.25, 5.0, 55)
tau_len = np.linspace(0.08, 1.0, 55)
Tr, Ta = np.meshgrid(t_reset, tau_len)
S_mat = Tr + Ta

Z_hjm = np.vectorize(caplet_black_vol_from_gaussian_hjm, otypes=[float])(Tr, S_mat, sigma0, kappa)

fig = plt.figure(figsize=(11, 5))
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
surf = ax1.plot_surface(Tr, Ta, Z_hjm, cmap=cm.coolwarm, linewidth=0, antialiased=True)
ax1.set_xlabel("Reset time T")
ax1.set_ylabel("Accrual τ = S − T")
ax1.set_zlabel("Black vol")
ax1.set_title("Gaussian HJM caplet Black vol")
fig.colorbar(surf, ax=ax1, shrink=0.55)

ax2 = fig.add_subplot(1, 2, 2)
cf = ax2.contourf(Tr, Ta, Z_hjm, levels=26, cmap="coolwarm")
ax2.set_xlabel("Reset time T")
ax2.set_ylabel("Accrual τ")
ax2.set_title("Contour view")
fig.colorbar(cf, ax=ax2)
plt.tight_layout()
plt.show()